# Per-image diagnostic — why DG3 fits some images better than others (ch04)

The figure ch04 includes is `results/per_image_diagnostic_initial_1003/scatter.png`:
all 1003 stimuli, the pretrained read-out, scored under the thesis protocol
(`--dataset-variant initial`). Computing it is a cluster job. **Redrawing it is
not** — section 1 below rebuilds it from the committed `diagnostic.json` in
seconds, with no model and no corpus, which is what makes the committed figure
checkable from a fresh clone.

Section 2 then runs the pipeline itself on a laptop-sized sample, under the same
protocol. That sample will *not* reproduce the committed figure — a different
number of stimuli gives different correlations — so it writes to a scratch
directory. It is there to show the computation, not to re-derive the artefact.

The question the diagnostic asks: the dataset-level number (IG ≈ 1.56 bits/fix)
hides a wide spread across images, from about −0.13 to 3.96. What predicts where
in that range an image lands? The candidate is how much the human subjects agreed
with each other, measured as the normalised Shannon entropy of the pooled
fixation density (`fixation_entropy_norm`, ch03 §Human variability). It runs
against every per-image metric and hardest against the log-likelihood
(ρ ≈ −0.84): DG3 is uncertain roughly where the human population is uncertain.

`diagnostic.json` also carries `centroid_spread_norm` and `consensus_area_75_pct`.
The thesis reports neither — fixation entropy is the one scalar it uses — but they
are still computed, and the correlations cell prints whatever the file holds.

> **Section 2 is the slow one.** It runs DG3 over every scanpath of every sampled
> stimulus. Measured at over 30 minutes on an M-series laptop over MPS at
> `N_STIM = 50`; `--n-stim 1003` is a cluster job, not a laptop one.

In [ ]:
import runpy
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
    REPO = REPO.parent

from IPython.display import Image, display


def run_script(name, *args):
    """Run scripts/<name> exactly as the command line would.

    These notebooks drive the same code the committed figure came from rather
    than reimplementing the plotting, so a notebook cannot silently disagree
    with what is in results/ and in the thesis.
    """
    sys.argv = [name, *[str(a) for a in args]]
    runpy.run_path(str(REPO / 'scripts' / name), run_name='__main__')


def show(*paths, width=1000):
    for p in paths:
        display(Image(filename=str(REPO / p), width=width))

## 2. Run the pipeline on a laptop-sized sample

### PARAMETERS — edit these

In [ ]:
N_STIM          = 50          # stimuli to evaluate; 1003 is the cluster run
SEED            = 0           # which stimuli get sampled
DATASET_VARIANT = 'initial'   # the thesis protocol (ch03 §The MIT1003 dataset)
START_FIXATION  = 1           # index 0 is history, not a target
OUT_DIR         = 'results/_scratch/per_image'   # scratch: a run overwrites its
                                                 # --out in place, and neither
                                                 # committed diagnostic should be
                                                 # overwritten by a 50-stimulus
                                                 # sample

The two settings that fix the protocol are `DATASET_VARIANT` and `START_FIXATION`,
and they only make sense together.

MIT1003 forces the first fixation of every scanpath to the screen centre. The
`initial` variant keeps that fixation at index 0, and `START_FIXATION = 1` then
uses it as history and scores every free fixation — 104,171 of them, the protocol
every arm in the thesis is trained and scored under. Under `plain`, `pysaliency`
has already dropped the centre on load, so the same `START_FIXATION = 1` skips the
first *free* fixation as well and scores 89,255; nothing in the thesis uses that
protocol. The 50-stimulus sample this notebook runs is a laptop-sized version of
the committed 1003-stimulus `results/per_image_diagnostic_initial_1003/`.

Scoring index 0 under either variant would plant an identical centre spike on
every image and flatter any model carrying a centerbias — which DG3 does, inside
its `Finalizer`.

In [ ]:
run_script('per_image_diagnostic.py',
           '--n-stim', N_STIM,
           '--seed', SEED,
           '--dataset-variant', DATASET_VARIANT,
           '--start-fixation', START_FIXATION,
           '--out', REPO / OUT_DIR)

show(f'{OUT_DIR}/scatter.png', width=1100)

In [ ]:
## The correlations, as numbers

Both files, side by side: the 1003-stimulus artefact ch04 reads, and the sample
just computed. The signs should agree; the magnitudes will not, and a 50-stimulus
correlation is loose enough that the gap is uninformative on its own.

In [ ]:
import json


def correlations(path):
    d = json.loads(Path(path).read_text())
    return d['correlations'], len(d['rows'])


thesis, n_thesis = correlations(REPO / THESIS_DIR / 'diagnostic.json')
sample, n_sample = correlations(REPO / OUT_DIR / 'diagnostic.json')

print(f"{'scalar':<24}{'metric':<10}{f'rho @ n={n_thesis}':>16}{f'rho @ n={n_sample}':>16}")
for scalar, metrics in thesis.items():
    for metric, stats in metrics.items():
        other = sample.get(scalar, {}).get(metric, {})
        print(f'{scalar:<24}{metric:<10}{stats["spearman"]:>16.3f}'
              f'{other.get("spearman", float("nan")):>16.3f}')

Each run also writes a readable summary next to the JSON, at
`<out>/diagnostic.md`.

## Command line

```bash
# redraw the ch04 figure from the committed artefact — no model, no corpus
.venv/bin/python scripts/per_image_diagnostic.py --replot \
    --out results/per_image_diagnostic_initial_1003

# a laptop-sized sample under the thesis protocol, into a scratch directory
.venv/bin/python scripts/per_image_diagnostic.py --n-stim 50 \
    --dataset-variant initial --out results/_scratch/per_image

# the run behind the committed artefact — cluster only
.venv/bin/python scripts/per_image_diagnostic.py --n-stim 1003 \
    --dataset-variant initial --out results/per_image_diagnostic_initial_1003
```

In [ ]:
import json
d = json.loads((REPO / 'results/per_image_diagnostic/diagnostic.json').read_text())
print(json.dumps(d.get('correlations', d), indent=2)[:2000])

The run also writes a readable summary at
`results/per_image_diagnostic/diagnostic.md`.

## Command line

```bash
.venv/bin/python scripts/per_image_diagnostic.py --n-stim 50
```